In [ ]:
import pandas as pd
import re
# Load data and data dictionary
data = pd.read_excel("va_data.xlsx", sheet_name="Sheet1")
dictionary = pd.read_excel("data_dictionary.xlsx", sheet_name="Sheet1")

In [ ]:
def create_demographic_intro(row):
    parts = []
    age = row.get('age')
    sex = row.get('sex')
    sex_text = None
    # Decode sex
    try:
      sex_str = str(int(float(sex)))
    except:
      sex_str = str(sex).strip().lower()

    sex_map = {
        "1": "female",
        "0": "male"
    }

    sex_text = sex_map.get(sex_str, sex_str)
    # Build sentence
    if not pd.isna(age) and sex_text:
        return f"The deceased was a {int(age)}-year-old {sex_text}."
    elif not pd.isna(age):
        return f"The deceased was {int(age)} years old."
    elif sex_text:
        return f"The deceased was {sex_text}."
    return ""

In [ ]:
# Convert coded values like "No=0, Yes=1, Unknown=8"
def parse_codes(code_string):
    if pd.isna(code_string):
        return {}
    pairs = re.findall(r'([^=,]+)=([^,]+)', str(code_string))
    return {v.strip(): k.strip() for k, v in pairs}

def clean_label(label):
    # remove numeric tags like "-2000 fever"
    label = re.sub(r"-?\d+\s*\w*$", "", label)
    return label.strip()

def describe_value(var, value):
    # Skip truly missing values
    if pd.isna(value):
        return None

    # Convert to string for checking
    value_str = str(value).strip().lower()

    # Skip blank or textual missing values
    if value_str in ["", "na", "nan", "none"]:
        return None

    info = dict_map.get(var)
    if not info:
        return None

    label = clean_label(info["label"])
    vtype = info["type"]
    codes = info["codes"]

    value_str = str(int(value)) if isinstance(value, float) and value.is_integer() else str(value)

    # Skip No, Unknown, Missing unless you want them included
    if value_str in ["8", "9","99", "999"]:
        return None
    # Special free-text fields
    if var in ["sex", "age"]:
      return None
    if var in ["vtelldiedtx", "vknowcodtx"]:
        text = str(value).strip()
        if text == "":
            return None
        return f"{label}: {text}."
    else:
      if vtype == "categorical":
          decoded = codes.get(value_str, value_str)
          if decoded.lower() == "yes":
              return f"The deceased had {label.lower()}."
          else:
              return f"{label} was {decoded.lower()}."
      elif vtype == "numeric":
          return f"{label}: {value}."
    return None


def create_narrative(row):
    sentences = []
    demo = create_demographic_intro(row)
    if demo:
      sentences.append(demo)
    for var in dict_map.keys():
      # Stop reading variables at narr_avail
      if var == "narr_avail":
        break
      if var in row:
        sentence = describe_value(var, row[var])
        if sentence:
          sentences.append(sentence)

    return " ".join(sentences)

In [ ]:
def create_missing_narrative(row):

    missing_items = []

    for var in dict_map.keys():

        # Stop at narr_avail
        if var == "narr_avail":
            break

        if var not in row:
            continue

        value = row[var]

        info = dict_map.get(var)
        if not info:
            continue

        label = clean_label(info["label"])

        # Missing values
        if pd.isna(value):
            missing_items.append(label)
            continue

        value_str = str(value).strip().lower()

        # Empty values
        if value_str in ["", "na", "nan", "none"]:
            missing_items.append(label)
            continue

        # Unknown / Missing codes
        if value_str in ["8", "9", "99", "999"]:
            missing_items.append(label)

    if len(missing_items) == 0:
        return ""

    return (
        "The following information was not recorded or is unknown: "
        + ", ".join(missing_items)
        + "."
    )

In [ ]:
dictionary["code_map"] = dictionary["value_labels"].apply(parse_codes)

dict_map = {
    row["variable"]: {
        "label": row["variable_desc"],
        "type": row["data_type"],
        "codes": row["code_map"]
    }
    for _, row in dictionary.iterrows()
}

# Create narratives
data["generated_narrative"] = data.apply(create_narrative, axis=1)


In [ ]:
# Missing / unknown narrative
data["missing_narrative"] = data.apply(create_missing_narrative, axis=1)

In [ ]:
# Keep only ID and narrative
output = data[[
    "ident",
    "narrative",
    "generated_narrative",
    "missing_narrative"
]]
# Save output
output.to_excel("va_narratives.xlsx", index=False)